In [ ]:
import os, logging
import subprocess

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
import networkx as nx
import vcfpy

from data import save_preprocessed_data, read_preprocessed_data, SNPDataSet

### Set variables

In [ ]:
raw_data_dir = '/home/share/data/1kGP'
preprocess_dir = os.path.join(raw_data_dir, "preprocessed")

chr_list = [str(x) for x in range(1,23)]
gt_dict = {"0|0" :0, "0|1" : 1, "1|0" : 1, "1|1" : 2} # convert string genotype -> integer of dosage 

sample_metadata_file = os.path.join(raw_data_dir, "igsr-1000 genomes 30x on grch38.tsv")
pedigree_file = os.path.join(raw_data_dir, "1kGP.3202_samples.pedigree_info.txt")

### Validation checks

In [ ]:
assert os.path.exists(raw_data_dir), f"Data path not exists: {raw_data_dir}"
assert os.path.isfile(sample_metadata_file), f"File not exists : {sample_metadata_file}"
assert os.path.isfile(pedigree_file), f"File not exists : {pedigree_file}"

if not os.path.exists(preprocess_dir):
    os.makedirs(preprocess_dir)

In [ ]:
def count_vcf_line_count(filename):
    cmd = f"grep -v '^#' {filename} | wc -l"
    result = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    return int(result.stdout.strip())

def preprocess_vcf(vcf_file_name, sample_metadata_file, output_file_prefix):
    vcf_line_count = count_vcf_line_count(vcf_file_name)
    print(f"----- processing {vcf_file_name} file with {vcf_line_count} variants -----")

    reader = vcfpy.Reader.from_path(vcf_file_name)
    sample_names_from_vcf = reader.header.samples.names

    gt_mat_raw = np.empty((vcf_line_count, len(sample_names_from_vcf)), dtype=np.int8)
    gt_mat_raw.fill(0)

    variant_info_header = ['CHROM', 'POS', 'REF', 'ALT']
    variant_info_df_raw = pd.DataFrame(index = range(vcf_line_count), columns = variant_info_header, dtype = str)

    status_counter = {
        "total_variant" : 0,
        "total_snps" : 0,
        "total_meaningful_snps" : 0
    }

    for record in tqdm(reader, total = vcf_line_count):
        status_counter["total_variant"] += 1
        if not record.is_snv():
            continue
        status_counter["total_snps"] += 1

        gt_int = [gt_dict[call.data.get('GT')] if call.data.get('GT') in gt_dict else len(gt_dict) for call in record.calls]
        if all(v == 0 for v in gt_int):
            continue
        status_counter["total_meaningful_snps"] += 1

        assert len(gt_int) == gt_mat_raw.shape[1]
        gt_mat_raw[status_counter["total_meaningful_snps"]-1, :] = gt_int

        variant_info = [record.CHROM, str(record.POS), record.REF] + [alt.value for alt in record.ALT]
        variant_info_df_raw.iloc[status_counter["total_meaningful_snps"]-1] = variant_info
        
    reader.close()

    assert(status_counter["total_variant"] == vcf_line_count)
    print(status_counter)

    sample_metadata_df = pd.read_csv(sample_metadata_file, sep="\t")
    sample_name_to_idx = {name : idx for idx, name in enumerate(sample_names_from_vcf)}
    indices_for_sort_sample = [sample_name_to_idx[name] for name in sample_metadata_df["Sample name"]]

    gt_mat = gt_mat_raw[:status_counter["total_meaningful_snps"], :].transpose()[indices_for_sort_sample,:]
    variant_info_df = variant_info_df_raw.iloc[:status_counter["total_meaningful_snps"]]

    save_preprocessed_data(gt_mat, variant_info_df, output_file_prefix)
    print(f"sample metadata (#samples, ) : {sample_metadata_df.shape}")

### Read each vcf file and convert to matrix format

In [ ]:
for chr in chr_list[::-1]:
    vcf_file_name = os.path.join(raw_data_dir, f"1kGP_high_coverage_Illumina.chr{chr}.filtered.SNV_INDEL_SV_phased_panel.vcf")
    if not os.path.exists(vcf_file_name):
        logging.warning(f"can not find vcf file for chromosome {chr}")
        continue

    preprocess_vcf(vcf_file_name, sample_metadata_file, os.path.join(preprocess_dir, f"chr{chr}"))

## Merge each chromosome data into single array

In [ ]:
merged_file_save_prefix = os.path.join(preprocess_dir, "merged")

In [ ]:
genotype_array_list, variant_info_df_list = [], []
for chr in chr_list:
    gt_array, variant_info_df = read_preprocessed_data(os.path.join(preprocess_dir, f"chr{chr}"))
    
    if gt_array is not None:
        genotype_array_list.append(gt_array)
        variant_info_df_list.append(variant_info_df)

genotype_array_combined = np.concatenate(genotype_array_list, axis=1)
variant_info_df_combined = pd.concat(variant_info_df_list, axis=0, ignore_index=True)
print(f"Combine result: genotype array with shape {genotype_array_combined.shape} and variant info with shape {variant_info_df_combined.shape}")

In [ ]:
save_preprocessed_data(genotype_array_combined, variant_info_df_combined, merged_file_save_prefix)

## Preprocess labels

In [ ]:
sample_metadata_df = pd.read_csv(sample_metadata_file, sep="\t")
print(f"Read sample metadata info with shape : {sample_metadata_df.shape}")

pedigree_df = pd.read_csv(pedigree_file, sep=r"\s+", dtype=str)
print(f"Read pedigree info with shape : {pedigree_df.shape}")

In [ ]:
# Column renaming
sample_metadata_df = sample_metadata_df.rename(columns={"Sample name": "sample_id", "Sex": "sex", "Population code": "population_code", "Superpopulation code" : "superpopulation_code"})
pedigree_df = pedigree_df.rename(columns={"sampleID": "sample_id", "fatherID": "father_id", "motherID": "mother_id"})

# Sanity check
assert set(sample_metadata_df["sample_id"]) == set(pedigree_df["sample_id"])
assert sample_metadata_df["sample_id"].nunique() == len(sample_metadata_df)
assert pedigree_df["sample_id"].nunique() == len(pedigree_df)

print(f'{set(pedigree_df["father_id"]) - set(sample_metadata_df["sample_id"])}')
print(f'{set(pedigree_df["mother_id"]) - set(sample_metadata_df["sample_id"])}')

for c in ["sex", "population_code", "superpopulation_code"]:
    print(f'label "{c}" has {sample_metadata_df[c].nunique()} unique labels : {sample_metadata_df[c].unique().tolist()}')

In [ ]:
def add_family_id(pedigree_df):
    graph = nx.Graph()
    graph.add_nodes_from(pedigree_df["sample_id"])  # 모든 sample을 node로 추가

    for row in pedigree_df.itertuples(index=False):
        if row.father_id != "0":
            graph.add_edge(row.sample_id, row.father_id)
        if row.mother_id != "0":
            graph.add_edge(row.sample_id, row.mother_id)

    sample_to_family = {}
    for i, family_members in enumerate(nx.connected_components(graph)):
        family_id = min(family_members)  #f"FAMILY_{i:04d}"
        sample_to_family.update({sample_id: family_id for sample_id in family_members})

    pedigree_df["family_id"] = pedigree_df["sample_id"].map(sample_to_family)
    return pedigree_df

In [ ]:
pedigree_df = add_family_id(pedigree_df)

In [ ]:
label_df = sample_metadata_df[["sample_id", "sex", "population_code", "superpopulation_code"]].merge(pedigree_df[["sample_id", "father_id", "mother_id", "family_id"]], on="sample_id", validate="one_to_one")

In [ ]:
label_file_path = os.path.join(preprocess_dir, "labels.csv")
label_df.to_csv(label_file_path, index=False)
print(f"saved preprocessed label to {label_file_path}")

## Supporting sample count filtering

In [ ]:
target_name = "merged"
preprocessed_file_prefix = os.path.join(preprocess_dir, target_name)

snp_dataset = SNPDataSet.from_file(preprocessed_file_prefix, sample_metadata_df)

In [ ]:
supporting_sample_count = (snp_dataset.genotype_array != 0).sum(axis = 0)

In [ ]:
sns.histplot(supporting_sample_count / snp_dataset.genotype_array.shape[0], bins=100, kde=True)  # 'bins' controls the number of bins, 'kde' adds a Kernel Density Estimate plot

#plt.xscale('log')
#plt.xlim(xmin, xmax)

plt.title('Histogram of Data')
plt.xlabel('Population Allele Frequency')
plt.ylabel('Frequency')

plt.show()

In [ ]:
supporting_count_filter = (supporting_sample_count >= 3) & (supporting_sample_count <= (snp_dataset.genotype_array.shape[0] - 3))
print(f"This filter will retain {supporting_count_filter.sum()} /", snp_dataset.genotype_array.shape[1], "variants")

In [ ]:
snp_dataset.filter_variant(supporting_count_filter, inplace = True)

In [ ]:
snp_dataset.save_data(os.path.join(preprocess_dir, f"{target_name}_support3"))

## [obsolete] Convert old genotype 0~3 into dosage genotype 0~2

In [ ]:
target_name = "merged_support3"
preprocessed_file_prefix = os.path.join(preprocess_dir, target_name)

gt_array, variant_info_df = read_preprocessed_data(preprocessed_file_prefix)

In [ ]:
print(f"Unique genotype values before conversion: {np.unique(gt_array)}")

gt_array[gt_array == 2] = 1
gt_array[gt_array == 3] = 2

print(f"Unique genotype values after conversion: {np.unique(gt_array)}")

In [ ]:
save_preprocessed_data(gt_array, variant_info_df, preprocessed_file_prefix)